## Business problem: How can CIDFF #Rhône-Arc Alpin identify emerging needs, gaps in existing services and potential areas for strategic development in supporting women affected by violence?

# 1. Data Exploration and Feasibility

## 1.1 Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup

## 1.2 Loading the Sexism Barometer dataset

The Sexism Barometer dataset is explored as a potential data source to analyse 
emerging gender-related issues, particularly sexism and masculinist attitudes.

In [ ]:
# Load the raw Sexism Barometer dataset
df = pd.read_csv(
    "base-de-donnees-toluna-harris-barometre-sexisme-vague-5-2025-dgcs-hce-.csv",
    sep=";"
)

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
list(df.columns)

In [ ]:
q4_19 = [col for col in df.columns if col.startswith("Q4_19")][0]
df[q4_19].unique()

In [ ]:
df[q4_19].value_counts()

In [ ]:
973 / 2779 * 100

In [ ]:
df["POIDS"].head()

In [ ]:
df["POIDS"].describe()

In [ ]:
df["POIDS"] = df["POIDS"].str.replace("," , ".").astype(float)
df["POIDS"].describe()

# 2. Mapping the Existing Support Ecosystem

The objective of this section is to identify organizations supporting women
affected by violence across the four departments covered by CIDFF Rhône-Arc Alpin:
Rhône (69), Isère (38), Savoie (73), and Haute-Savoie (74).

The analysis aims to explore which organizations are present, the issues they
address, and, where available, the services they provide.

## 2.1 Data Source: Arrêtons les violences Directory

The national "Arrêtons les violences" directory is explored as a web source
to identify specialized organizations operating within the study area.

The first step is to examine the structure of the website and determine
which information can be collected through web scraping.

## 2.2 Inspecting the Web Page

Before collecting the data, the structure of the directory is inspected to understand
how the information is displayed and how it can be extracted.

The study focuses on the four departments covered by CIDFF Rhône-Arc Alpin:
Rhône (69), Isère (38), Savoie (73), and Haute-Savoie (74).

In [ ]:
url = "https://annuaire.arretonslesviolences.gouv.fr/search?q=Lyon&lat=45.758&lng=4.835&label=Lyon"
response = requests.get(url)
response.status_code

In [ ]:
print(response.text[ :1000])

In [ ]:
print("CIDFF" in response.text.upper())

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")
texts = soup.stripped_strings
matches = [text for text in texts if "cidff" in text.lower()]
print (matches[:10])

In [ ]:
association_name = soup.find(string="Cidff Rhône - Arcalpin")

print(association_name.parent)

In [ ]:
headings = soup.find_all("h3")

print(len(headings))
print([heading.get_text(strip=True) for heading in headings[:10]])

In [ ]:
# Inspect the HTML structure around the first association name
print(headings[0].parent.prettify()[:2000])

In [ ]:
# Inspect the larger container around the first association
print(headings[0].parent.parent.prettify()[:4000])

In [ ]:
# Select the first association card
card = headings[0].parent.parent

# Extract the association name
name = card.find("h3").get_text(strip=True)

# Extract the email address
email_link = card.find("a", href=lambda href: href and href.startswith("mailto:"))
email = email_link.get_text(strip=True) if email_link else None

print(name)
print(email)

In [ ]:
# Extract information from all association cards
associations = []

for heading in headings:
    card = heading.parent.parent

    name = heading.get_text(strip=True)

    email_link = card.find("a", href=lambda href: href and href.startswith("mailto:"))
    email = email_link.get_text(strip=True) if email_link else None

    website_link = card.find("a", href=lambda href: href and href.startswith("http"))
    website = website_link.get("href") if website_link else None

    tags = card.select("p.fr-tag")
    violence_types = [tag.get_text(strip=True) for tag in tags]

    associations.append({
        "name": name,
        "email": email,
        "website": website,
        "violence_types": violence_types
    })

# Convert the extracted data into a DataFrame
df = pd.DataFrame(associations)

display(df)

In [ ]:
# Inspect the text content of the second association card
card = headings[1].parent.parent

print(card.get_text(" | ", strip=True))

In [ ]:
# Extract detailed information from association cards
associations = []

for heading in headings:
    card = heading.parent.parent
    details = {}

    # Match each information label with its value
    for dt in card.find_all("dt"):
        dd = dt.find_next_sibling("dd")
        if dd:
            details[dt.get_text(strip=True)] = dd.get_text(" ", strip=True)

    # Extract the association name and network
    name = heading.get_text(strip=True)
    network_tag = card.select_one("p.fr-hint-text")
    network = network_tag.get_text(" ", strip=True).replace("Réseau :", "").strip() if network_tag else None

    # Extract contact details
    phone = details.get("Téléphone")
    email = details.get("Adresse e-mail")

    website_link = card.find("a", href=lambda href: href and href.startswith("http"))
    website = website_link.get("href") if website_link else None

    # Extract violence categories
    violence_types = [tag.get_text(strip=True) for tag in card.select("p.fr-tag")]

    associations.append({
        "name": name,
        "network": network,
        "phone": phone,
        "email": email,
        "website": website,
        "violence_types": violence_types
    })

# Create the updated DataFrame
df = pd.DataFrame(associations)

display(df)

In [ ]:
# Save the first extracted dataset as a CSV file
df.to_csv("associations_lyon_test.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Find links that may lead to additional search results
links = soup.find_all("a", href=True)

for link in links:
    href = link.get("href")
    if "page=" in href or "search?" in href:
        print(link.get_text(" ", strip=True), ":", href)

In [ ]:
# Request the second page of search results
url_page2 = "https://annuaire.arretonslesviolences.gouv.fr/search?q=Lyon&lat=45.758&lng=4.835&label=Lyon&page=2"

response_page2 = requests.get(url_page2)
soup_page2 = BeautifulSoup(response_page2.text, "html.parser")

# Extract association names from page 2
headings_page2 = soup_page2.find_all("h3")

print("Status:", response_page2.status_code)
print("Associations:", len(headings_page2))
print([h.get_text(strip=True) for h in headings_page2])

In [ ]:
# Collect association data from all three search result pages
associations = []

base_url = "https://annuaire.arretonslesviolences.gouv.fr/search"

for page in range(1, 4):
    # Request and parse each page
    params = {
        "q": "Lyon",
        "lat": 45.758,
        "lng": 4.835,
        "label": "Lyon",
        "page": page
    }

    response = requests.get(base_url, params=params, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    headings = soup.find_all("h3")

    # Extract information from each association card
    for heading in headings:
        card = heading.parent.parent
        details = {}

        for dt in card.find_all("dt"):
            dd = dt.find_next_sibling("dd")
            if dd:
                details[dt.get_text(strip=True)] = dd.get_text(" ", strip=True)

        network_tag = card.select_one("p.fr-hint-text")
        network = (
            network_tag.get_text(" ", strip=True).replace("Réseau :", "").strip()
            if network_tag else None
        )

        website_link = card.find(
            "a", href=lambda href: href and href.startswith("http")
        )

        associations.append({
            "name": heading.get_text(strip=True),
            "network": network,
            "phone": details.get("Téléphone"),
            "email": details.get("Adresse e-mail"),
            "website": website_link.get("href") if website_link else None,
            "violence_types": [
                tag.get_text(strip=True) for tag in card.select("p.fr-tag")
            ],
            "search_page": page
        })

    print(f"Page {page}: {len(headings)} associations")

# Create the combined dataset
df = pd.DataFrame(associations)

print(f"\nTotal: {len(df)} association records")
display(df.head())

In [ ]:
# Save the Lyon search results before further processing
df.to_csv("associations_lyon_raw.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Identify records sharing the same email address
duplicates = df[df["email"].notna() & df.duplicated(subset=["email"], keep=False)]

display(duplicates[["name", "email"]])

In [ ]:
# Test the Grenoble search results
grenoble_url = "https://annuaire.arretonslesviolences.gouv.fr/search"

grenoble_params = {
    "q": "Grenoble",
    "lat": 45.182828,
    "lng": 5.724301,
    "label": "Grenoble"
}

response_grenoble = requests.get(
    grenoble_url, params=grenoble_params, timeout=20
)
response_grenoble.raise_for_status()

soup_grenoble = BeautifulSoup(response_grenoble.text, "html.parser")
headings_grenoble = soup_grenoble.find_all("h3")

print("Associations on page 1:", len(headings_grenoble))
print([h.get_text(strip=True) for h in headings_grenoble])

In [ ]:
# Find pagination links in the Grenoble search results
for link in soup_grenoble.find_all("a", href=True):
    href = link.get("href")

    if "page=" in href or "search?" in href:
        print(link.get_text(" ", strip=True), ":", href)

In [ ]:
# Collect association names from all Grenoble result pages
def scrape_city_names(city, lat, lng, total_pages):
    names = []

    for page in range(1, total_pages + 1):
        params = {
            "q": city,
            "lat": lat,
            "lng": lng,
            "label": city,
            "page": page
        }

        response = requests.get(
            "https://annuaire.arretonslesviolences.gouv.fr/search",
            params=params,
            timeout=20
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        headings = soup.find_all("h3")

        names.extend([h.get_text(strip=True) for h in headings])

        print(f"{city} - Page {page}: {len(headings)} records")

    return names


# Test the function with Grenoble
grenoble_names = scrape_city_names(
    "Grenoble", 45.182828, 5.724301, 2
)

print(f"\nTotal Grenoble records: {len(grenoble_names)}")

In [ ]:
# Extract association details from all pages of a city search
def scrape_city(city, lat, lng, total_pages):
    associations = []
    base_url = "https://annuaire.arretonslesviolences.gouv.fr/search"

    for page in range(1, total_pages + 1):
        params = {
            "q": city,
            "lat": lat,
            "lng": lng,
            "label": city,
            "page": page
        }

        response = requests.get(base_url, params=params, timeout=20)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for heading in soup.find_all("h3"):
            card = heading.parent.parent
            details = {}

            # Match each contact label with its value
            for dt in card.find_all("dt"):
                dd = dt.find_next_sibling("dd")
                if dd:
                    details[dt.get_text(strip=True)] = dd.get_text(" ", strip=True)

            # Extract the association network
            network_tag = card.select_one("p.fr-hint-text")
            network = (
                network_tag.get_text(" ", strip=True)
                .replace("Réseau :", "").strip()
                if network_tag else None
            )

            # Extract the website URL
            website_link = card.find(
                "a", href=lambda href: href and href.startswith("http")
            )

            associations.append({
                "name": heading.get_text(strip=True),
                "network": network,
                "phone": details.get("Téléphone"),
                "email": details.get("Adresse e-mail"),
                "website": website_link.get("href") if website_link else None,
                "violence_types": [
                    tag.get_text(strip=True) for tag in card.select("p.fr-tag")
                ],
                "search_city": city,
                "search_page": page
            })

        print(f"{city} - Page {page}: completed")

    return pd.DataFrame(associations)

In [ ]:
# Collect and combine results from both cities
df_lyon = scrape_city("Lyon", 45.758, 4.835, 3)
df_grenoble = scrape_city("Grenoble", 45.182828, 5.724301, 2)

df_combined = pd.concat([df_lyon, df_grenoble], ignore_index=True)

print("Total records:", len(df_combined))
display(df_combined.groupby("search_city").size())

In [ ]:
# Save the combined raw dataset
df_combined.to_csv(
    "associations_lyon_grenoble_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# Check pagination links for Chambéry and Annecy
cities = {
    "Chambéry": (45.583223, 5.909299),
    "Annecy": (45.901584, 6.125296)
}

for city, (lat, lng) in cities.items():
    params = {
        "q": city,
        "lat": lat,
        "lng": lng,
        "label": city
    }

    response = requests.get(
        "https://annuaire.arretonslesviolences.gouv.fr/search",
        params=params,
        timeout=20
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Display pagination links and the number of cards on page 1
    print(f"\n{city}: {len(soup.find_all('h3'))} records on page 1")

    for link in soup.find_all("a", href=True):
        if "page=" in link["href"]:
            print(link.get_text(" ", strip=True), ":", link["href"])

In [ ]:
# Collect association records for Savoie and Haute-Savoie
df_chambery = scrape_city("Chambéry", 45.583223, 5.909299, 3)
df_annecy = scrape_city("Annecy", 45.901584, 6.125296, 2)

In [ ]:
# Combine all city search results into one dataset
df_all = pd.concat(
    [df_lyon, df_grenoble, df_chambery, df_annecy],
    ignore_index=True
)

# Check the number of records by search city
print("Total records:", len(df_all))
display(df_all.groupby("search_city").size())

In [ ]:
# Save the raw dataset before cleaning
df_all.to_csv(
    "associations_4_cities_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# Check duplicates using columns without Python lists
columns_to_check = [
    "name", "network", "phone", "email", "website"
]

print(
    "Duplicate records:",
    df_all.duplicated(subset=columns_to_check).sum()
)

# Find all records sharing the same association name
repeated_names = df_all[
    df_all.duplicated(subset=["name"], keep=False)
]

display(
    repeated_names[
        ["name", "search_city", "email"]
    ].sort_values("name")
)

In [ ]:
# Display records with identical contact information
duplicate_groups = df_all[
    df_all.duplicated(
        subset=["name", "network", "phone", "email", "website"],
        keep=False
    )
]

print("Records in duplicate groups:", len(duplicate_groups))

display(
    duplicate_groups[
        ["name", "search_city", "phone", "email"]
    ].sort_values(["name", "search_city"])
)

### Geographic Coverage and Duplicates

We collected 87 records from searches around Lyon, Grenoble, Chambéry and Annecy.

The directory uses proximity-based searches, so these results may not cover all associations operating in the four departments.

Some associations operate in several departments and may share the same contact details.

Our goal is to count each association once per department, while keeping track of its local offices.

Before removing duplicates, we need to verify where each association operates. For now, we keep all 87 records.

In [ ]:
# Inspect links in the first association card
card = soup.find("h3").parent.parent

for link in card.find_all("a"):
    print(link.get_text(" ", strip=True), "→", link.get("href"))

In [ ]:
# Inspect the HTML attributes of the association card
print(card.attrs)
print(card.parent.attrs)

In [ ]:
# Find scripts containing geographic information
for script in soup.find_all("script"):
    content = script.get_text()
    if any(word in content.lower() for word in ["latitude", "longitude", "coordinates"]):
        print(content[:1000])       

In [ ]:
# Search for association data inside scripts
for script in soup.find_all("script"):
    content = script.get_text()
    if "mfpf74@hotmail.fr" in content:
        position = content.find("mfpf74@hotmail.fr")
        print(content[max(0, position-500):position+1000])

In [ ]:
# Count association IDs in the embedded data
import re

script_text = " ".join(
    script.get_text() for script in soup.find_all("script")
)

ids = re.findall(r'\\"id\\":(\d+)', script_text)

print("IDs found:", len(ids))
print("First IDs:", ids[:10])

In [ ]:
# Count unique IDs in the embedded data
print("Total IDs:", len(ids))
print("Unique IDs:", len(set(ids)))

In [ ]:
# Check whether the embedded data includes other departments
print("Paris:", "75002 Paris" in script_text)
print("Le Mans:", "72000 Le Mans" in script_text)

In [ ]:
# Find scripts containing association IDs
for i, script in enumerate(soup.find_all("script")):
    content = script.get_text()
    count = len(re.findall(r'\\"id\\":(\d+)', content))

    if count > 0:
        print(f"Script {i}: {count} IDs")

In [ ]:
# Inspect the beginning of the data script
data_script = soup.find_all("script")[22].get_text()

print("Script length:", len(data_script))
print(data_script[:500])

In [ ]:
import json

# Extract and decode the Next.js data
raw_data = data_script.split("self.__next_f.push([1,", 1)[1].rsplit("])", 1)[0]
decoded_data = json.loads(raw_data)

print(type(decoded_data))
print(decoded_data[:300])

In [ ]:
import json

# Decode the embedded Next.js data
raw_data = data_script.split("self.__next_f.push([1,", 1)[1].rsplit("])", 1)[0]
decoded_data = json.loads(raw_data)

print(type(decoded_data))
print(decoded_data[:300])

In [ ]:
# Find the first association record
position = decoded_data.find('"postalCode"')

print("Position:", position)
print(decoded_data[position - 350:position + 450])

In [ ]:
# Locate the association list
start = decoded_data.find('24:[') + len('24:')

# Extract the JSON array
decoder = json.JSONDecoder()
associations, end = decoder.raw_decode(decoded_data[start:])

print("Number of records:", len(associations))
print("First record:", associations[0]["name"])
print("Available columns:", associations[0].keys())

In [ ]:
# Create a DataFrame from the extracted records
df = pd.DataFrame(associations)

# Extract department codes from postal codes
df["department"] = df["postalCode"].astype("string").str[:2]

# Keep the four departments of our study
departments = ["69", "38", "73", "74"]
df_territory = df[df["department"].isin(departments)].copy()

print("Total records:", len(df))
print("Records in our territory:", len(df_territory))
print("\nRecords by department:")
print(df_territory["department"].value_counts().sort_index())

In [ ]:
# Compare association names across both datasets
raw_names = set(df_all["name"].dropna().str.strip().str.lower())
new_names = set(df_territory["name"].dropna().str.strip().str.lower())

missing_names = sorted(raw_names - new_names)

print("Unique names in initial scraping:", len(raw_names))
print("Unique names in new dataset:", len(new_names))
print("Names absent from new dataset:", len(missing_names))

print("\nExamples:")
print(missing_names[:15])

In [ ]:
# Extract IDs from the current page
ids_page_1 = set(re.findall(r'\\"id\\":(\d+)', data_script))

# Extract IDs from another search page
other_soup = BeautifulSoup(requests.get(
    "https://annuaire.arretonslesviolences.gouv.fr/search",
    params={"q": "Grenoble", "page": 1}
).text, "html.parser")

other_scripts = " ".join(
    script.get_text() for script in other_soup.find_all("script")
)

ids_page_2 = set(re.findall(r'\\"id\\":(\d+)', other_scripts))

print("IDs on first page:", len(ids_page_1))
print("IDs on Grenoble page:", len(ids_page_2))
print("New IDs:", len(ids_page_2 - ids_page_1))

In [ ]:
# Search missing associations in all 833 records
df_missing = df[
    df["name"].str.strip().str.lower().isin(missing_names)
]

print(df_missing[["name", "city", "postalCode"]].to_string(index=False))

In [ ]:
# Check missing geographical information
print("Total records:", len(df))
print("Missing postal codes:", df["postalCode"].isna().sum())
print("Missing cities:", df["city"].isna().sum())
print("Missing coordinates:", df["latitude"].isna().sum())

In [ ]:
# Find records with coordinates but no postal code
df_geo_missing = df[
    df["postalCode"].isna()
    & df["latitude"].notna()
    & df["longitude"].notna()
]

print("Records:", len(df_geo_missing))

display(
    df_geo_missing[
        ["name", "city", "postalCode", "latitude", "longitude"]
    ].head(15)
)

In [ ]:
# Check the records of interest
names_to_check = ["Filactions", "Ça Démenage", "Savoie De Femmes"]

display(
    df[df["name"].isin(names_to_check)][
        ["name", "address", "postalCode", "latitude", "longitude", "id"]
    ]
)

In [ ]:
# Find the nearest address using GPS coordinates
# The API returns geographic information in GeoJSON format

url = "https://data.geopf.fr/geocodage/reverse"

params = {
    "lat": 45.773409,
    "lon": 4.836649
}

response = requests.get(url, params=params, timeout=15)
response.raise_for_status()

data = response.json()

print("Number of results:", len(data["features"]))

if data["features"]:
    print(data["features"][0]["properties"])

In [ ]:
# Find the department from GPS coordinates
# IGN is France's national mapping and geographic information agency

def get_department(lat, lon):
    url = "https://data.geopf.fr/geocodage/reverse"

    params = {
        "lat": lat,
        "lon": lon,
        "limit": 1
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()

        features = response.json().get("features", [])

        if features:
            # Extract the department code from the postal code
            postcode = features[0]["properties"].get("postcode")

            if postcode:
                return postcode[:2]

    except requests.RequestException:
        return None

    return None

In [ ]:
# Test the function with Filactions
print(get_department(45.773409, 4.836649))

In [ ]:
import time

# Keep the original department code when available
df["department"] = df["postalCode"].astype("string").str[:2]

# Select records with missing postal codes but valid GPS coordinates
mask = (
    df["department"].isna()
    & df["latitude"].notna()
    & df["longitude"].notna()
)

# Find the department for each selected record
for idx, row in df.loc[mask].iterrows():
    df.at[idx, "department"] = get_department(
        row["latitude"], row["longitude"]
    )

    # Add a short delay between API requests
    time.sleep(0.2)

# Keep records located in our four departments
target_codes = ["69", "38", "73", "74"]
df_territory = df[df["department"].isin(target_codes)].copy()

# Check the new territorial coverage
print("Total records:", len(df_territory))
print("\nRecords by department:")
print(df_territory["department"].value_counts().sort_index())

In [ ]:
# Check the dataset size and number of unique records
print("Dataset shape:", df_territory.shape)
print("Unique IDs:", df_territory["id"].nunique())
print("Duplicate IDs:", df_territory["id"].duplicated().sum())

# Check missing values in the main columns
columns_to_check = [
    "name", "address", "city", "postalCode",
    "latitude", "longitude", "network",
    "violenceTypes"
]

# Keep only columns that exist in the dataset
columns_to_check = [
    col for col in columns_to_check
    if col in df_territory.columns
]

print("\nMissing values:")
print(df_territory[columns_to_check].isna().sum())

In [ ]:
# Check available fields and their completeness
print("Available columns:")
print(df_territory.columns.tolist())

# Check how many records have at least one violence category
print("\nRecords with violence categories:")
print(df_territory["violenceTypes"].apply(
    lambda x: isinstance(x, list) and len(x) > 0
).value_counts())

# Check the number of distinct organizations
print("\nUnique organization names:")
print(df_territory["name"].nunique())

In [ ]:
df_inclusion = pd.read_csv(
    "../structures-inclusion-2026-10-12.csv",
    low_memory=False
)

print(df_inclusion.shape)
display(df_inclusion.head())

In [ ]:
# Select structures in our four departments
departments = ["38", "69", "73", "74"]

df_inclusion_local = df_inclusion[
    df_inclusion["code_insee"].astype("string").str[:2].isin(departments)
].copy()

print("Total structures:", len(df_inclusion_local))

display(
    df_inclusion_local["code_insee"]
    .astype("string")
    .str[:2]
    .value_counts()
    .sort_index()
)

In [ ]:
# Search for known organisations
names = [
    "VIFFIL",
    "LE MAS",
    "SCJE",
    "TANDEM",
    "AMICALE DU NID",
    "MOUVEMENT DU NID",
    "ALURS",
    "DÉMÉNAGE"
]

results = df_inclusion_local[
    df_inclusion_local["nom"].str.contains(
        "|".join(names),
        case=False,
        na=False,
        regex=True
    )
]

display(results[["nom", "commune", "source", "description"]])

In [ ]:
# Search for VSS/VIF-related terms
keywords = r"violences conjugales|violences intrafamiliales|violences faites aux femmes|violences sexuelles|victimes de violences"

text_data = (
    df_inclusion_local["nom"].fillna("") + " " +
    df_inclusion_local["description"].fillna("")
)

df_inclusion_vss = df_inclusion_local[
    text_data.str.contains(keywords, case=False, na=False)
].copy()

print("Structures matching VSS/VIF keywords:", len(df_inclusion_vss))
display(df_inclusion_vss[["nom", "commune", "source"]].head(30))

In [ ]:
%pip install pymupdf

In [ ]:
import pymupdf

pdf_path = "../data/raw/cartographie_acteurs_lyon_2025.pdf"

doc = pymupdf.open(pdf_path)

print("Number of pages:", len(doc))
print("\nExtracted text preview:\n")
print(doc[0].get_text()[:3000])

In [ ]:
# Check text extraction on each page
for page_number, page in enumerate(doc, start=1):
    text = page.get_text()
    print(f"Page {page_number}: {len(text)} characters")
    print(text[:400])
    print("-" * 50)

In [ ]:
# Extract the full text of the map
map_text = doc[5].get_text()

print(map_text)

In [ ]:
# Organisations identified in the Lyon ecosystem map
map_organisations = [
    "VIFFIL",
    "LE MAS",
    "SCJE",
    "CIDFF",
    "AFCCC"
]

print("Organisations to check:", len(map_organisations))
print(map_organisations)

In [ ]:
# Check available RNA columns
print(df_rna.columns.tolist())

In [ ]:
# Check the current directory dataset
print("Total records:", len(df_territory))
print("Unique organisations IDs:", df_territory["id"].nunique())

print("\nRecords by department:")
print(df_territory["department"].value_counts(dropna=False))

print("\nAvailable columns:")
print(df_territory.columns.tolist())

In [ ]:
from pathlib import Path

# Find CSV files in the project folder
project_folder = Path("..")

for file in project_folder.rglob("*.csv"):
    print(file.relative_to(project_folder))